In [1]:
import random
import torch
import os
import numpy as np
import pandas as pd
import polars as pl

In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
def make_aggregated_outputs(input_file, metrics = ['accuracy', 'precision', 'recall', 'f1', 'kappa', 'MCC']):
    dimensions = [
        'informational_vs_involved',
        'non-narrative_vs_narrative',
        'situation-dependent_vs_explicit',
        'non-persuasive_vs_persuasive',
        'non-abstract_vs_abstract',
        'compressed_vs_elaborated'
    ]

    saved_output_dict = {}

    for folder in os.listdir(INPUT_DIR):
        folder_path = os.path.join(INPUT_DIR, folder)

        if not (os.path.isdir(folder_path) and 'outputs' in folder):
            continue

        saved_output_dict[folder] = {
            dim: {metric: [] for metric in metrics}
            for dim in dimensions
        }

        for sub_folder in os.listdir(folder_path):
            sub_path = os.path.join(folder_path, sub_folder)

            if not (os.path.isdir(sub_path) and sub_folder.isdigit()):
                continue

            file_path = os.path.join(sub_path, f"{input_file}.csv")
            df = pd.read_csv(file_path)

            for dimension in dimensions:
                temp_df = df[df['dimension'] == dimension]

                if temp_df.empty:
                    raise ValueError(f"There should be something in the temp_df for the dimension {dimension}.")

                row = temp_df.iloc[0]

                for metric in metrics:
                    saved_output_dict[folder][dimension][metric].append(float(row[metric]))

    temp_dict_all = {}

    for folder, dimensions_dict in saved_output_dict.items():
        series_list = []

        for dimension, metrics_dict in dimensions_dict.items():
            mean_series = pd.Series({
                metric: (sum(values) / len(values)) if values else float('nan')
                for metric, values in metrics_dict.items()
            }, name=dimension)

            series_list.append(mean_series)

        df_folder = pd.concat(series_list, axis=1)
        temp_dict_all[folder] = df_folder

    df_all_folders = pd.concat(temp_dict_all, axis=0)

    df_mean_all = df_all_folders.groupby(level=1).mean()

    return df_all_folders, df_mean_all

In [5]:
all_classif, mean_classif = make_aggregated_outputs('classification_comparison_results_zero_vs_biber')

In [6]:
all_classif

informational_vs_involved  non-narrative_vs_narrative  \
outputsTest  accuracy                    0.608800                    0.553100   
             precision                   0.519761                    0.450990   
             recall                      0.668924                    0.562160   
             f1                          0.584283                    0.499533   
             kappa                       0.225858                    0.104419   
             MCC                         0.232724                    0.106815   
outputsAll   accuracy                    0.611900                    0.562700   
             precision                   0.524449                    0.465142   
             recall                      0.678213                    0.582968   
             f1                          0.590573                    0.515966   
             kappa                       0.233108                    0.126972   
             MCC                         0.240820                    0.130188   
outputsTrain accuracy                    0.610900                    0.567300   
             precision                   0.523189                    0.470600   
             recall                      0.681349                    0.590390   
             f1                          0.590809                    0.522372   
             kappa                       0.232452                    0.136529   
             MCC                         0.240687                    0.139994   

                        situation-dependent_vs_explicit  \
outputsTest  accuracy                          0.612900   
             precision                         0.671610   
             recall                            0.647951   
             f1                                0.658743   
             kappa                             0.210987   
             MCC                               0.211772   
outputsAll   accuracy                          0.621700   
             precision                         0.679829   
             recall                            0.653548   
             f1                                0.665583   
             kappa                             0.230180   
             MCC                               0.231192   
outputsTrain accuracy                          0.616200   
             precision                         0.674748   
             recall                            0.650836   
             f1                                0.661836   
             kappa                             0.217487   
             MCC                               0.218332   

                        non-persuasive_vs_persuasive  \
outputsTest  accuracy                       0.557000   
             precision                      0.414054   
             recall                         0.587310   
             f1                             0.484532   
             kappa                          0.116171   
             MCC                            0.121975   
outputsAll   accuracy                       0.561400   
             precision                      0.409264   
             recall                         0.581756   
             f1                             0.479263   
             kappa                          0.120181   
             MCC                            0.126486   
outputsTrain accuracy                       0.566400   
             precision                      0.415446   
             recall                         0.577179   
             f1                             0.481923   
             kappa                          0.125989   
             MCC                            0.131784   

                        non-abstract_vs_abstract  compressed_vs_elaborated  
outputsTest  accuracy                   0.520800                  0.469500  
             precision                  0.249621                  0.124380  
             recall                     0.598996                  

In [7]:
mean_classif

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MCC,0.238077,0.125666,0.220432,0.126748,0.090469,-0.136079
accuracy,0.610533,0.561033,0.616933,0.561600,0.525467,0.471200
f1,0.588555,0.512624,0.662054,0.481906,0.355197,0.183128
kappa,0.230472,0.122640,0.219551,0.120780,0.073260,-0.111105
precision,0.522466,0.462244,0.675396,0.412921,0.252577,0.129289
recall,0.676162,0.578506,0.650778,0.582081,0.606847,0.319783


In [8]:
all_contin, mean_contin = make_aggregated_outputs('continuous_comparison_results_zero_vs_biber', ['pearson', 'spearman', 'MSE', 'RMSE', 'MAE'])

In [9]:
all_contin

informational_vs_involved  non-narrative_vs_narrative  \
outputsTest  pearson                    0.288506                    0.069623   
             spearman                   0.302424                    0.121451   
             MSE                        1.422987                    1.860753   
             RMSE                       1.191375                    1.362373   
             MAE                        0.959578                    1.033800   
outputsAll   pearson                    0.304339                    0.085816   
             spearman                   0.315726                    0.141426   
             MSE                        1.391321                    1.828368   
             RMSE                       1.177586                    1.350434   
             MAE                        0.951817                    1.021618   
outputsTrain pearson                    0.307755                    0.096096   
             spearman                   0.320366                    0.148911   
             MSE                        1.384490                    1.807807   
             RMSE                       1.174278                    1.342822   
             MAE                        0.950186                    1.018568   

                       situation-dependent_vs_explicit  \
outputsTest  pearson                          0.126719   
             spearman                         0.233215   
             MSE                              1.746562   
             RMSE                             1.319335   
             MAE                              0.989424   
outputsAll   pearson                          0.160874   
             spearman                         0.260718   
             MSE                              1.678253   
             RMSE                             1.293117   
             MAE                              0.970289   
outputsTrain pearson                          0.145740   
             spearman                         0.239487   
             MSE                              1.708519   
             RMSE                             1.305260   
             MAE                              0.981889   

                       non-persuasive_vs_persuasive  non-abstract_vs_abstract  \
outputsTest  pearson                       0.120701                  0.073446   
             spearman                      0.150267                  0.132173   
             MSE                           1.758597                  1.853108   
             RMSE                          1.324253                  1.359156   
             MAE                           0.995459                  1.037643   
outputsAll   pearson                       0.126225                  0.063404   
             spearman                      0.151511                  0.127158   
             MSE                           1.747551                  1.873193   
             RMSE                          1.320208                  1.366412   
             MAE                           0.994201                  1.035409   
outputsTrain pearson                       0.118398                  0.078876   
             spearman                      0.159170                  0.140558   
             MSE                           1.763204                  1.842248   
             RMSE                          1.326190                  1.355363   
             MAE                           0.989035                  1.030209   

                       compressed_vs_elaborated  
outputsTest  pearson                  -0.071306  
             spearman                 -0.151496  
             MSE                       2.142613  
             RMSE                      1.462673  
             MAE                       1.078184  
outputsAll   pearson                  -0.074666  
             spearman                 -0.137390  
             MSE                       2.149332  
             RMSE                      1.464687  
             MAE

In [10]:
mean_contin

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MAE,0.953860,1.024662,0.980534,0.992898,1.034420,1.075346
MSE,1.399599,1.832309,1.711111,1.756451,1.856183,2.149056
RMSE,1.181080,1.351876,1.305904,1.323550,1.360311,1.464590
pearson,0.300200,0.083845,0.144444,0.121775,0.071909,-0.074528
spearman,0.312839,0.137263,0.244473,0.153649,0.133296,-0.144286
